Predict agitation in the dataset Galaxy_records.

Let's see if we can pinpoint agitated clients. 

In [1]:
# Environment setup
import os

def check_environment():
    try:
        import google.colab
        return "Google Colab"
    except ImportError:
        return "Local Environment"

env = check_environment()
if env == "Google Colab":
    print("Running in Google Colab")
    !pip install -q datasets
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.chdir('/content/drive/My Drive/Colab Notebooks/GenCareAI/scripts/200_agitation_models')
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    print("Running in Local Environment")
    # !pip install -q 
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

Running in Local Environment


In [9]:
import pandas as pd
import joblib
from datasets import load_dataset
import spacy
from gensim.models import LdaMulticore, Word2Vec
import numpy as np

In [4]:
# Omgevingsvariabelen laden
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
PATH_HF_DATASET = 'ekrombouts/Galaxy_records'

# Dataset laden van Hugging Face
dataset = load_dataset(PATH_HF_DATASET, token=HF_TOKEN)

# Dataset converteren naar een pandas DataFrame
df = pd.DataFrame(dataset['train'])  # Aangenomen dat de data in de 'train' split zit

In [5]:
# Modellen en preprocessing hulpmiddelen laden
lda_model_path = "../../models/agitation/lda_model"
word2vec_model_path = "../../models/agitation/word2vec_model"
lda_model = LdaMulticore.load(lda_model_path)
word2vec_model = Word2Vec.load(word2vec_model_path)

# Dutch SpaCy model laden voor preprocessing
nlp = spacy.load('nl_core_news_sm', disable=['parser', 'tagger', 'ner'])

# Functie voor tekst preprocessing: lowercasing, tokenization, lemmatization, stopwoordverwijdering, en woordselectie op basis van part-of-speech
def preprocess_text(text, nlp_model):
    doc = nlp_model(text)
    cleaned_tokens = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop and token.pos_ in ['VERB', 'NOUN', 'ADJ', 'ADV', 'INTJ']]
    return " ".join(cleaned_tokens)

In [7]:
# Preprocessing toepassen op de dataset
df['text_clean'] = df['note'].apply(lambda x: preprocess_text(x, nlp))

In [10]:
# Topic distributies berekenen op basis van het LDA-model
def get_topic_distributions(lda_model, text, dictionary):
    bow = dictionary.doc2bow(text.split())
    doc_topics = lda_model.get_document_topics(bow, minimum_probability=0)
    topic_distribution = {f'topic_{i}': 0 for i in range(lda_model.num_topics)}
    for topic, prob in doc_topics:
        topic_distribution[f'topic_{topic}'] = prob
    return pd.Series(topic_distribution)

# Woord embeddings berekenen op basis van het Word2Vec-model
def calculate_document_embedding(text, model):
    embeddings = [model.wv[word] for word in text.split() if word in model.wv]
    if not embeddings:
        return pd.Series(np.zeros(model.vector_size))
    mean_embedding = np.mean(embeddings, axis=0)
    return pd.Series(mean_embedding)

# Topic modellen en woord embeddings toevoegen aan de DataFrame
dictionary = lda_model.id2word
df[[f'topic_{i}' for i in range(lda_model.num_topics)]] = df['text_clean'].apply(lambda x: get_topic_distributions(lda_model, x, dictionary))
df[[f'embedding_{i}' for i in range(word2vec_model.vector_size)]] = df['text_clean'].apply(lambda x: calculate_document_embedding(x, word2vec_model))



In [11]:

# Modellen voor voorspelling laden
rf_model_path = "../../models/agitation/random_forest_model.joblib"
lr_model_path = "../../models/agitation/logistic_regression_model.joblib"
rf_model = joblib.load(rf_model_path)
lr_model = joblib.load(lr_model_path)

# Voorbereiden van de features voor voorspelling
feature_columns = [col for col in df.columns if col.startswith('topic_') or col.startswith('embedding_')]
X_new = df[feature_columns]

# Voorspellingen maken
df['rf_proba'] = rf_model.predict_proba(X_new)[:, 1]  # Kans op klasse 1 (agitation) met Random Forest
df['lr_proba'] = lr_model.predict_proba(X_new)[:, 1]  # Kans op klasse 1 (agitation) met Logistic Regression

# Resultaten bekijken
print(df.head())

   ct_id  month  iteration  day   time  \
0      1      1          1    1  08:00   
1      1      1          1    1  12:00   
2      1      1          1    1  16:00   
3      1      1          1    2  08:00   
4      1      1          1    2  12:00   

                                                note  \
0  Meneer heeft vanochtend hulp gehad bij het aan...   
1  Tijdens de lunch had meneer wat hulp nodig bij...   
2  In de middag heeft meneer deelgenomen aan een ...   
3  Vanmorgen had meneer wat hulp nodig bij de toi...   
4  Tijdens de lunch was meneer wat vergeetachtig ...   

                                          text_clean   topic_0   topic_1  \
0  meneer vanochtend hulp aankleden gaan ontbijte...  0.409940  0.010994   
1  lunch meneer hulp nodig snijden eten rustig ge...  0.011944  0.109524   
2  middag meneer deelnemen puzzelactiviteit erg c...  0.013063  0.457167   
3  vanmorgen meneer hulp nodig toiletgang wassen ...  0.417524  0.011922   
4  lunch meneer vergeetachtig 

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29668 entries, 0 to 29667
Data columns (total 66 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ct_id         29668 non-null  int64  
 1   month         29668 non-null  int64  
 2   iteration     29668 non-null  int64  
 3   day           29668 non-null  int64  
 4   time          29668 non-null  object 
 5   note          29668 non-null  object 
 6   text_clean    29668 non-null  object 
 7   topic_0       29668 non-null  float32
 8   topic_1       29668 non-null  float32
 9   topic_2       29668 non-null  float32
 10  topic_3       29668 non-null  float32
 11  topic_4       29668 non-null  float32
 12  topic_5       29668 non-null  float32
 13  topic_6       29668 non-null  float32
 14  embedding_0   29668 non-null  float32
 15  embedding_1   29668 non-null  float32
 16  embedding_2   29668 non-null  float32
 17  embedding_3   29668 non-null  float32
 18  embedding_4   29668 non-nu

In [13]:
# Optioneel: Sla de resultaten op
df.to_csv("../../data/predictions_with_probabilities.csv", index=False)
print("Voorspellingen opgeslagen in predictions_with_probabilities.csv")

Voorspellingen opgeslagen in predictions_with_probabilities.csv


In [17]:
print(df[['note', 'rf_proba', 'lr_proba']].sort_values(by='lr_proba', ascending=False).head(20))

                                                    note  rf_proba  lr_proba
13998  Tijdens de activiteit werd meneer heel agressi...      0.99  0.999073
23298  Meneer reageert niet op vragen en staart wezen...      0.25  0.998932
9249   Dhr werd tijdens de lunch opgewonden en begon ...      0.93  0.998912
3121   Mw begon te schreeuwen tijdens de activiteit. ...      0.53  0.998469
2779   Meneer werd plotseling onrustig en begon te sc...      0.90  0.998101
24005  Tijdens de activiteit vanmiddag geagiteerd ged...      0.97  0.997387
7191   Tijdens de groepsactiviteit vanmiddag begon me...      0.98  0.997378
27879  Meneer begon onrustig te worden en raakte geïr...      0.86  0.997019
8514   Dhr werd onrustig tijdens activiteit. Teruggeb...      0.36  0.995842
11609  Meneer Jansen kijkt constant uit het raam en m...      1.00  0.995824
9566   Meneer was gefrustreerd en gooide met spullen ...      0.96  0.995677
591    Na de ochtendactiviteiten leek mevrouw meer ve...      0.58  0.995638

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29668 entries, 0 to 29667
Data columns (total 66 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ct_id         29668 non-null  int64  
 1   month         29668 non-null  int64  
 2   iteration     29668 non-null  int64  
 3   day           29668 non-null  int64  
 4   time          29668 non-null  object 
 5   note          29668 non-null  object 
 6   text_clean    29668 non-null  object 
 7   topic_0       29668 non-null  float32
 8   topic_1       29668 non-null  float32
 9   topic_2       29668 non-null  float32
 10  topic_3       29668 non-null  float32
 11  topic_4       29668 non-null  float32
 12  topic_5       29668 non-null  float32
 13  topic_6       29668 non-null  float32
 14  embedding_0   29668 non-null  float32
 15  embedding_1   29668 non-null  float32
 16  embedding_2   29668 non-null  float32
 17  embedding_3   29668 non-null  float32
 18  embedding_4   29668 non-nu

In [22]:
print(df.groupby('ct_id')['lr_proba'].mean().sort_values(ascending=False))
print(df.groupby('ct_id')['rf_proba'].mean().sort_values(ascending=False))

ct_id
75    0.152228
26    0.146730
51    0.139907
47    0.122344
30    0.122164
        ...   
63    0.032947
25    0.030593
84    0.029333
88    0.028136
86    0.013118
Name: lr_proba, Length: 96, dtype: float64
ct_id
51    0.194622
75    0.182370
76    0.175111
19    0.155111
26    0.153048
        ...   
82    0.044333
39    0.039444
71    0.036311
94    0.033067
86    0.025068
Name: rf_proba, Length: 96, dtype: float64
